In [ ]:
import pandas as pd
import numpy as np
import uuid
from datetime import datetime, timedelta

# 실제 데이터 불러오기
df1 = pd.read_csv('/content/DEVICE_MANAGEMENT.csv')

# 날짜/시간 열이 문자열이라면 datetime 객체로 변환
if not pd.api.types.is_datetime64_any_dtype(df1['check_in']):
    df1['check_in'] = pd.to_datetime(df1['check_in'])
if not pd.api.types.is_datetime64_any_dtype(df1['check_out']):
    df1['check_out'] = pd.to_datetime(df1['check_out'])

# 결과 데이터프레임을 저장할 리스트
result_data = []

# 각 행에 대해 처리
for _, row in df1.iterrows():
    device_id = row['device_id']
    emp_id = row['emp_id']
    check_in = row['check_in']
    check_out = row['check_out']

    # 10분 간격으로 시간 생성
    current_time = check_in
    interval = timedelta(minutes=10)

    # 초기 배터리 값
    battery = 100

    # 배터리 감소율 계산 (시작: 100%, 종료: 약 20-30%)
    total_seconds = (check_out - check_in).total_seconds()
    if total_seconds <= 0:
        continue  # 체크인/체크아웃 시간이 이상한 경우 건너뛰기

    total_intervals = int(total_seconds / interval.total_seconds())
    battery_decrease_per_interval = (100 - 25) / total_intervals if total_intervals > 0 else 0

    while current_time <= check_out:
        # 측정 ID 생성
        measurement_id = str(uuid.uuid4())[:10]

        # 배터리 계산 (점진적 감소)
        battery = max(int(battery - battery_decrease_per_interval), 0)

        # 랜덤 값 생성 (일반 범위 내)
        hr_normal = np.random.randint(60, 100)
        temp_normal = round(np.random.uniform(36.0, 37.5), 1)
        resp_normal = np.random.randint(12, 20)
        spo2_normal = np.random.randint(95, 100)

        # 이상치 추가 (약 5% 확률로)
        if np.random.random() < 0.05:
            # 심박수 이상치
            hr = np.random.choice([np.random.randint(30, 50), np.random.randint(110, 180)])
        else:
            hr = hr_normal

        if np.random.random() < 0.05:
            # 체온 이상치
            temp = np.random.choice([round(np.random.uniform(34.0, 35.5), 1),
                                    round(np.random.uniform(38.0, 40.0), 1)])
        else:
            temp = temp_normal

        if np.random.random() < 0.05:
            # 호흡수 이상치
            resp = np.random.choice([np.random.randint(5, 10), np.random.randint(25, 40)])
        else:
            resp = resp_normal

        if np.random.random() < 0.05:
            # 산소포화도 이상치
            spo2 = np.random.randint(80, 94)
        else:
            spo2 = spo2_normal

        # 가속도 및 자이로 데이터 (일부 이상치 포함)
        acc_x = round(np.random.uniform(-2.0, 2.0), 2)
        acc_y = round(np.random.uniform(-2.0, 2.0), 2)
        acc_z = round(np.random.uniform(-2.0, 2.0), 2)

        gyro_x = round(np.random.uniform(-180, 180), 2)
        gyro_y = round(np.random.uniform(-180, 180), 2)
        gyro_z = round(np.random.uniform(-180, 180), 2)

        # 이상치를 위한 가속도 스파이크 (약 2% 확률)
        if np.random.random() < 0.02:
            acc_multiplier = np.random.randint(5, 15)
            acc_x *= acc_multiplier
            acc_y *= acc_multiplier
            acc_z *= acc_multiplier

        # 행 추가
        result_data.append({
            'measurement_id': measurement_id,
            'emp_id': emp_id,
            'device_id': device_id,
            'measure_time': current_time,
            'battery': battery,
            'hr': hr,
            'temp': temp,
            'resp': resp,
            'spo2': spo2,
            'acc_x': acc_x,
            'acc_y': acc_y,
            'acc_z': acc_z,
            'gyro_x': gyro_x,
            'gyro_y': gyro_y,
            'gyro_z': gyro_z,
            'heat_risk': None,  # 온열질환 위험도는 Null
            'fall_risk': None   # 낙상 위험도는 Null
        })

        # 다음 시간으로 이동
        current_time += interval

# 최종 데이터프레임 생성
df = pd.DataFrame(result_data)

# MySQL INSERT 문 생성 함수
def generate_insert_statements(df, table_name='DEVICE_MEASUREMENT'):
    insert_statements = []

    for _, row in df.iterrows():
        columns = []
        values = []

        for column, value in row.items():
            columns.append(column)

            # 값 타입에 따른 포맷팅
            if value is None:
                values.append('NULL')
            elif isinstance(value, (int, float)):
                if pd.isna(value):  # NaN 체크
                    values.append('NULL')
                else:
                    values.append(str(value))
            elif isinstance(value, pd.Timestamp) or isinstance(value, datetime):
                values.append(f"'{value.strftime('%Y-%m-%d %H:%M:%S')}'")
            else:
                # 문자열에 따옴표 이스케이프 처리
                escaped_value = str(value).replace("'", "\\'")
                values.append(f"'{escaped_value}'")

        # INSERT 문 생성
        columns_str = ', '.join(columns)
        values_str = ', '.join(values)
        insert_statement = f"INSERT INTO {table_name} ({columns_str}) VALUES ({values_str});"
        insert_statements.append(insert_statement)

    return insert_statements

# MySQL INSERT 문 생성
insert_statements = generate_insert_statements(df)

# 터미널에 출력 (샘플 10개만 보여주기)
print("MySQL INSERT 문 샘플 (처음 10개):")
for stmt in insert_statements[:10]:
    print(stmt)

# 텍스트 파일로 저장
output_file = '/content/mysql_insert_statements.txt'
with open(output_file, 'w') as f:
    for stmt in insert_statements:
        f.write(stmt + '\n')

print(f"\n전체 INSERT 문이 {output_file}에 저장되었습니다.")
print(f"총 {len(insert_statements)}개의 INSERT 문이 생성되었습니다.")

MySQL INSERT 문 샘플 (처음 10개):
INSERT INTO DEVICE_MEASUREMENT (measurement_id, emp_id, device_id, measure_time, battery, hr, temp, resp, spo2, acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z, heat_risk, fall_risk) VALUES ('32d27083-0', 'WR0009', 'DEV0030', '2025-04-01 06:00:00', 98, 96, 36.4, 12, 96, -0.31, -1.29, 1.86, -71.24, -68.06, 88.47, NULL, NULL);
INSERT INTO DEVICE_MEASUREMENT (measurement_id, emp_id, device_id, measure_time, battery, hr, temp, resp, spo2, acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z, heat_risk, fall_risk) VALUES ('2ca5f667-a', 'WR0009', 'DEV0030', '2025-04-01 06:10:00', 96, 91, 37.3, 12, 97, 0.17, 0.42, -1.11, 134.79, -146.13, -37.79, NULL, NULL);
INSERT INTO DEVICE_MEASUREMENT (measurement_id, emp_id, device_id, measure_time, battery, hr, temp, resp, spo2, acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z, heat_risk, fall_risk) VALUES ('3ca4d0b5-c', 'WR0009', 'DEV0030', '2025-04-01 06:20:00', 94, 75, 36.2, 16, 97, -1.83, -1.49, 0.28, -51.83, 55.63, 80.18, NULL, NULL);
INS

In [ ]:
csv_filename = '/content/device_measure.csv'
df.to_csv(csv_filename, index=False)

In [ ]:
import pandas as pd
import numpy as np
import uuid
from datetime import datetime

# 1. 측정 데이터 불러오기
device_measure_df = pd.read_csv('/content/device_measure.csv')

# 날짜 열이 문자열이라면 datetime 객체로 변환
if not pd.api.types.is_datetime64_any_dtype(device_measure_df['measure_time']):
    device_measure_df['measure_time'] = pd.to_datetime(device_measure_df['measure_time'])

# 2. 이상치 감지 기준 설정
anomaly_criteria = {
    'hr': {'low': 50, 'high': 100},         # 심박수 이상 기준 (50 미만, 100 초과)
    'temp': {'low': 35.5, 'high': 37.5},    # 체온 이상 기준 (35.5도 미만, 37.5도 초과)
    'resp': {'low': 10, 'high': 20},        # 호흡수 이상 기준 (10 미만, 20 초과)
    'spo2': {'low': 94, 'high': 100},       # 산소포화도 이상 기준 (94 미만)
    'acc': {'threshold': 10}                # 가속도 이상 기준 (벡터 크기가 10 초과)
}

# 3. 이상치 감지 함수
def detect_anomalies(df, criteria):
    anomalies = []

    for idx, row in df.iterrows():
        anomaly_detected = False
        symptom = []
        risk_type = ""

        # 심박수 확인
        if row['hr'] < criteria['hr']['low']:
            symptom.append(f"낮은 심박수 ({row['hr']} bpm)")
            anomaly_detected = True
            risk_type = "심장 관련 위험"
        elif row['hr'] > criteria['hr']['high']:
            symptom.append(f"높은 심박수 ({row['hr']} bpm)")
            anomaly_detected = True
            risk_type = "심장 관련 위험"

        # 체온 확인
        if row['temp'] < criteria['temp']['low']:
            symptom.append(f"저체온 ({row['temp']}°C)")
            anomaly_detected = True
            risk_type = "체온 관련 위험"
        elif row['temp'] > criteria['temp']['high']:
            symptom.append(f"고열 ({row['temp']}°C)")
            anomaly_detected = True
            risk_type = "체온 관련 위험"

        # 호흡수 확인
        if row['resp'] < criteria['resp']['low']:
            symptom.append(f"낮은 호흡수 ({row['resp']} 회/분)")
            anomaly_detected = True
            risk_type = "호흡 관련 위험"
        elif row['resp'] > criteria['resp']['high']:
            symptom.append(f"높은 호흡수 ({row['resp']} 회/분)")
            anomaly_detected = True
            risk_type = "호흡 관련 위험"

        # 산소포화도 확인
        if row['spo2'] < criteria['spo2']['low']:
            symptom.append(f"낮은 산소포화도 ({row['spo2']}%)")
            anomaly_detected = True
            risk_type = "저산소증 위험"

        # 가속도 벡터 크기 계산 및 확인
        acc_magnitude = np.sqrt(row['acc_x']**2 + row['acc_y']**2 + row['acc_z']**2)
        if acc_magnitude > criteria['acc']['threshold']:
            symptom.append(f"갑작스러운 움직임 감지 (가속도: {acc_magnitude:.2f})")
            anomaly_detected = True
            risk_type = "낙상 위험"

        # 이상 현상이 감지되면 기록
        if anomaly_detected:
            # 이상 현상 ID 생성
            anomaly_id = str(uuid.uuid4())[:10]

            # 위치 정보 생성 (실제 데이터에서는 이 부분이 다를 수 있음)
            # 예시로 가속도 값을 기반으로 임의 위치 생성
            loc_x = row['acc_x'] * 10 if not pd.isna(row['acc_x']) else np.random.uniform(0, 100)
            loc_y = row['acc_y'] * 10 if not pd.isna(row['acc_y']) else np.random.uniform(0, 100)

            anomalies.append({
                'anomaly_id': anomaly_id,
                'emp_id': row['emp_id'],
                'anomaly_time': row['measure_time'],
                'symptom': '; '.join(symptom),
                'risk': risk_type,
                'loc_x': loc_x,
                'loc_y': loc_y
            })

    return pd.DataFrame(anomalies)

# 4. 이상치 감지 실행
anomaly_df = detect_anomalies(device_measure_df, anomaly_criteria)

# 5. 결과 출력
print(f"감지된 이상 현상 수: {len(anomaly_df)}")
print("\n이상 현상 데이터 샘플:")
print(anomaly_df.head())

# 6. MySQL INSERT 문 생성 함수
def generate_insert_statements(df, table_name='HEALTH_ANOMALY'):
    insert_statements = []

    for _, row in df.iterrows():
        columns = []
        values = []

        for column, value in row.items():
            columns.append(column)

            # 값 타입에 따른 포맷팅
            if value is None:
                values.append('NULL')
            elif isinstance(value, (int, float)):
                if pd.isna(value):  # NaN 체크
                    values.append('NULL')
                else:
                    values.append(str(value))
            elif isinstance(value, pd.Timestamp) or isinstance(value, datetime):
                values.append(f"'{value.strftime('%Y-%m-%d %H:%M:%S')}'")
            else:
                # 문자열에 따옴표 이스케이프 처리
                escaped_value = str(value).replace("'", "\\'")
                values.append(f"'{escaped_value}'")

        # INSERT 문 생성
        columns_str = ', '.join(columns)
        values_str = ', '.join(values)
        insert_statement = f"INSERT INTO {table_name} ({columns_str}) VALUES ({values_str});"
        insert_statements.append(insert_statement)

    return insert_statements

# 7. MySQL INSERT 문 생성
insert_statements = generate_insert_statements(anomaly_df)

# 8. INSERT 문 저장
output_file = '/content/health_anomaly_inserts.txt'
with open(output_file, 'w') as f:
    for stmt in insert_statements:
        f.write(stmt + '\n')

print(f"\nINSERT 문이 {output_file}에 저장되었습니다.")
print(f"총 {len(insert_statements)}개의 INSERT 문이 생성되었습니다.")

# 9. CSV 파일로도 저장
anomaly_df.to_csv('/content/health_anomaly.csv', index=False)
print("\n이상 현상 데이터가 'health_anomaly.csv'에 저장되었습니다.")

감지된 이상 현상 수: 991

이상 현상 데이터 샘플:
   anomaly_id  emp_id        anomaly_time  \
0  95b3d92a-c  WR0009 2025-04-01 06:30:00   
1  f83e8881-8  WR0009 2025-04-01 07:50:00   
2  392c940e-b  WR0009 2025-04-01 09:00:00   
3  d3ff736b-d  WR0009 2025-04-01 10:10:00   
4  46160c5e-a  WR0009 2025-04-01 11:00:00   

                                            symptom      risk  loc_x  loc_y  
0                   낮은 심박수 (42 bpm); 낮은 산소포화도 (88%)   저산소증 위험    2.9   14.8  
1                                    낮은 호흡수 (9 회/분)  호흡 관련 위험  -13.1   17.7  
2                                      저체온 (34.9°C)  체온 관련 위험    0.5  -12.4  
3                                    낮은 산소포화도 (90%)   저산소증 위험    2.7   -8.7  
4  낮은 심박수 (34 bpm); 높은 호흡수 (32 회/분); 낮은 산소포화도 (90%)   저산소증 위험   15.6   10.5  

INSERT 문이 /content/health_anomaly_inserts.txt에 저장되었습니다.
총 991개의 INSERT 문이 생성되었습니다.

이상 현상 데이터가 'health_anomaly.csv'에 저장되었습니다.
